## Setup

In [181]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
import pysam
import pyranges as pr

In [5]:
# Read in datasets
real1 = pd.read_csv('../data/real1/snv-parse-real1-labeled.txt', sep='\t', dtype={'Chr': str})
real2 = pd.read_csv('../data/real2_part1/snv-parse-real2_part1-labeled.txt', sep='\t', dtype={'Chr': str})
syn1 = pd.read_csv('../data/syn1/snv-parse-syn1-labeled.txt', sep='\t', dtype={'Chr': str})
syn2 = pd.read_csv('../data/syn2/snv-parse-syn2-labeled.txt', sep='\t', dtype={'Chr': str})
syn3 = pd.read_csv('../data/syn3/snv-parse-syn3-labeled.txt', sep='\t', dtype={'Chr': str})
syn4 = pd.read_csv('../data/syn4/snv-parse-syn4-labeled.txt', sep='\t', dtype={'Chr': str})
syn5 = pd.read_csv('../data/syn5/snv-parse-syn5-labeled.txt', sep='\t', dtype={'Chr': str})


In [ ]:
# Check dataset for range and NaN values
full_dataset = pd.concat([real1, real2, syn1, syn2, syn3, syn4, syn5], ignore_index=True)
print(full_dataset.describe())
print("\nNaN values by column:")
print(full_dataset.isna().sum())  # Check NaN count per column


       START_POS_REF   END_POS_REF          m2_MQ         f_MQMR  \
count   3.062210e+05  3.062210e+05  125584.000000  109252.000000   
mean    7.331304e+07  7.331304e+07      57.496172      54.963817   
std     5.726451e+07  5.726451e+07       5.267026      12.546463   
min     3.000000e+00  3.000000e+00       7.860000       0.000000   
25%     2.637710e+07  2.637710e+07      57.990000      58.714300   
50%     6.169157e+07  6.169157e+07      60.000000      60.000000   
75%     1.109767e+08  1.109767e+08      60.000000      60.000000   
max     2.492405e+08  2.492405e+08      63.640000      70.000000   

              vs_SSC         vs_SPV         vd_SSF         vd_MSI  
count  238759.000000  238759.000000  161303.000000  161303.000000  
mean       17.264593       0.117079       0.086558       3.084829  
std        18.236466       0.164220       0.179495       3.816062  
min         0.000000       0.000000       0.000000       0.000000  
25%         7.000000       0.006809       0.000

## Defining test and train dataset

In [35]:
# Define the training dataset
train_dataset = pd.concat([real1, syn1, syn2, syn3, syn4, syn5], ignore_index=True)

true_count = (train_dataset["True_SNV"] == True).sum()
false_count = (train_dataset["True_SNV"] == False).sum()

print(f"Total True values: {true_count}")
print(f"Total False values: {false_count}")

train_dataset.head()

Total True values: 75766
Total False values: 207855


,Chr,START_POS_REF,END_POS_REF,REF,ALT,REF_MFVdVs,ALT_MFVdVs,Sample_Name,FILTER_Mutect2,FILTER_Freebayes,FILTER_Vardict,FILTER_Varscan,m2_MQ,f_MQMR,vs_SSC,vs_SPV,vd_SSF,vd_MSI,True_SNV
0,1,13110,13110,G,A,G/NA/G/G/,A/NA/A/A/,icgc_cll-T,True,False,False,False,41.91,NaN,2.0,0.522430,0.23427,2.0,False
1,1,15015,15015,G,C,G/NA/NA/G/,C/NA/NA/C/,icgc_cll-T,True,False,False,False,43.42,NaN,5.0,0.302390,NaN,NaN,False
2,1,16949,16949,A,C,NA/NA/NA/A/,NA/NA/NA/C/,icgc_cll-T,False,False,False,True,NaN,NaN,16.0,0.023282,NaN,NaN,False
3,1,40552,40552,T,C,NA/NA/NA/T/,NA/NA/NA/C/,icgc_cll-T,False,False,False,True,NaN,NaN,26.0,0.002231,NaN,NaN,False
4,1,46907,46907,T,C,NA/NA/NA/T/,NA/NA/NA/C/,icgc_cll-T,False,False,False,True,NaN,NaN,17.0,0.017670,NaN,NaN,False


In [31]:
# Define the testing dataset
test_dataset= pd.concat([real2], ignore_index=True)

true_count = (test_dataset["True_SNV"] == True).sum()
false_count = (test_dataset["True_SNV"] == False).sum()

print(f"Total True values: {true_count}")
print(f"Total False values: {false_count}")

test_dataset.head()

Total True values: 449
Total False values: 22151


,Chr,START_POS_REF,END_POS_REF,REF,ALT,REF_MFVdVs,ALT_MFVdVs,Sample_Name,FILTER_Mutect2,FILTER_Freebayes,FILTER_Vardict,FILTER_Varscan,m2_MQ,f_MQMR,vs_SSC,vs_SPV,vd_SSF,vd_MSI,True_SNV
0,1,10291,10291,C,T,NA/NA/C/C/,NA/NA/T/T/,icgc_mbl-T,False,False,True,False,NaN,NaN,2.0,0.627830,0.00000,6.0,False
1,1,10330,10330,C,A,NA/NA/NA/C/,NA/NA/NA/A/,icgc_mbl-T,False,False,False,True,NaN,NaN,24.0,0.003559,NaN,NaN,False
2,1,10700,10700,G,C,NA/G/NA/NA/,NA/C/NA/NA/,icgc_mbl-T,False,True,False,False,NaN,3.88679,NaN,NaN,NaN,NaN,False
3,1,12719,12719,G,C,NA/NA/NA/G/,NA/NA/NA/C/,icgc_mbl-T,False,False,False,True,NaN,NaN,13.0,0.043528,NaN,NaN,False
4,1,14574,14574,A,G,A/NA/A/A/,G/NA/G/G/,icgc_mbl-T,False,False,True,True,34.81,NaN,23.0,0.004372,0.03642,2.0,False


In [134]:
# Select the feature columns 
# feature_columns = ["FILTER_Mutect2", "FILTER_Freebayes", "FILTER_Vardict", "FILTER_Varscan"]
# feature_columns = ["FILTER_Mutect2", "FILTER_Freebayes", "FILTER_Vardict", "FILTER_Varscan", "f_MQMR"]
feature_columns = ["FILTER_Mutect2", "FILTER_Freebayes", "FILTER_Vardict", "FILTER_Varscan", "m2_MQ", "f_MQMR", "vs_SSC", "vs_SPV", "vd_SSF", "vd_MSI"]
X_train = train_dataset[feature_columns]
X_train.head()

,FILTER_Mutect2,FILTER_Freebayes,FILTER_Vardict,FILTER_Varscan,m2_MQ,f_MQMR,vs_SSC,vs_SPV,vd_SSF,vd_MSI
0,True,False,False,False,41.91,NaN,2.0,0.522430,0.23427,2.0
1,True,False,False,False,43.42,NaN,5.0,0.302390,NaN,NaN
2,False,False,False,True,NaN,NaN,16.0,0.023282,NaN,NaN
3,False,False,False,True,NaN,NaN,26.0,0.002231,NaN,NaN
4,False,False,False,True,NaN,NaN,17.0,0.017670,NaN,NaN


In [135]:
# Select the target columns
target_columns = ["True_SNV"]
y_train = train_dataset[target_columns]
y_train.head()

,True_SNV
0,False
1,False
2,False
3,False
4,False


In [ ]:
# Define the test X and y
X_test = test_dataset[feature_columns]
y_test = test_dataset[target_columns]

## Feature preprocessing

In [158]:
# Define the function for feature preprocessing
def preprocess_features(df):
    df = df.replace([np.inf, -np.inf], 0).fillna(0)
    # df = df.reset_index()
    # df = df.astype(np.float32)
    # print(np.any(np.isnan(df)))
    # print(np.any(np.isinf(df)))
    # print(np.all(np.isfinite(df)))
    # print(df.dtypes)
    return df

In [177]:
# Define the function for target proprocessing
def proprocess_target(df):
    return df.squeeze()

In [139]:
# Proprocess X_train
X_train_processed = preprocess_features(X_train)
X_train_processed.head()

False
False
True
FILTER_Mutect2         bool
FILTER_Freebayes       bool
FILTER_Vardict         bool
FILTER_Varscan         bool
m2_MQ               float64
f_MQMR              float64
vs_SSC              float64
vs_SPV              float64
vd_SSF              float64
vd_MSI              float64
dtype: object


,FILTER_Mutect2,FILTER_Freebayes,FILTER_Vardict,FILTER_Varscan,m2_MQ,f_MQMR,vs_SSC,vs_SPV,vd_SSF,vd_MSI
0,True,False,False,False,41.91,0.0,2.0,0.522430,0.23427,2.0
1,True,False,False,False,43.42,0.0,5.0,0.302390,0.00000,0.0
2,False,False,False,True,0.00,0.0,16.0,0.023282,0.00000,0.0
3,False,False,False,True,0.00,0.0,26.0,0.002231,0.00000,0.0
4,False,False,False,True,0.00,0.0,17.0,0.017670,0.00000,0.0


## DecisionTreeClassifier

In [140]:
# Initialize and train the decision tree classifier.
clf = DecisionTreeClassifier(random_state=88)
clf.fit(X_train_processed, y_train)

DecisionTreeClassifier(random_state=88)

In [ ]:
# Predict using the trained models
y_predicted = clf.predict(preprocess_features(X_test))

# Obtain scores
print(classification_report(y_test, y_predicted))


False
False
True
FILTER_Mutect2         bool
FILTER_Freebayes       bool
FILTER_Vardict         bool
FILTER_Varscan         bool
m2_MQ               float64
f_MQMR              float64
vs_SSC              float64
vs_SPV              float64
vd_SSF              float64
vd_MSI              float64
dtype: object
              precision    recall  f1-score   support

       False       0.99      0.99      0.99     22151
        True       0.74      0.73      0.74       449

    accuracy                           0.99     22600
   macro avg       0.87      0.86      0.87     22600
weighted avg       0.99      0.99      0.99     22600



In [143]:
# Visualise the trained decision tree
# plt.figure(figsize=(12, 8))
# plot_tree(clf, feature_names=feature_columns, class_names=["False", "True"], filled=True)
# plt.title("Decision Tree for SNV Truth Prediction")
# plt.show()

## RandomForestClassifier

In [145]:
# Initialize and train the random forest classifier
rf_clf = RandomForestClassifier(random_state=88)
rf_clf.fit(X_train_processed, y_train)

/var/folders/07/vncpj_sd6mn0tb677n6ckp3c0000gn/T/ipykernel_42286/2703739492.py:3: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  rf_clf.fit(X_train_processed, y_train)


RandomForestClassifier(random_state=88)

In [147]:
# Predict using the trained models
y_predicted = rf_clf.predict(preprocess_features(X_test))

# Obtain scores
print(classification_report(y_test, y_predicted))

False
False
True
FILTER_Mutect2         bool
FILTER_Freebayes       bool
FILTER_Vardict         bool
FILTER_Varscan         bool
m2_MQ               float64
f_MQMR              float64
vs_SSC              float64
vs_SPV              float64
vd_SSF              float64
vd_MSI              float64
dtype: object
              precision    recall  f1-score   support

       False       1.00      1.00      1.00     22151
        True       0.88      0.78      0.83       449

    accuracy                           0.99     22600
   macro avg       0.94      0.89      0.91     22600
weighted avg       0.99      0.99      0.99     22600



## Training on 80% of combined dataset, testing on remaining 20%

In [183]:
# Split each dataset into respective test and train sets
split_data = {}
datasets = [real1, real2, syn1, syn2, syn3, syn4, syn5]
dataset_names = ["real1", "real2", "syn1", "syn2", "syn3", "syn4", "syn5"]

for name, df in zip(dataset_names, datasets):
    X = df[feature_columns]  
    y = df[target_columns]   
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=88, stratify=y
    )
    
    split_data[name] = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    }

In [184]:
# Concatenate the train sets
X_train_all = pd.concat([split_data[name]["X_train"] for name in split_data], axis=0)
y_train_all = pd.concat([split_data[name]["y_train"] for name in split_data], axis=0)

In [191]:
# Concatenate the test sets
X_test_all = pd.concat([split_data[name]["X_test"] for name in split_data], axis=0)
y_test_all = pd.concat([split_data[name]["y_test"] for name in split_data], axis=0)

In [187]:
# Training and testing the decision tree classifier
clf = DecisionTreeClassifier(random_state=88)
clf.fit(preprocess_features(X_train_all), proprocess_target(y_train_all))

print("DecisionTreeClassifier")
for name in split_data:
    y_pred = clf.predict(preprocess_features(split_data[name]["X_test"]))
    f1 = f1_score(split_data[name]["y_test"], y_pred)
    precision = precision_score(split_data[name]["y_test"], y_pred)
    recall = recall_score(split_data[name]["y_test"], y_pred)

    # Print formatted results
    print(f"{name}: F1 Score = {f1:.3f}, Precision = {precision:.3f}, Recall = {recall:.3f}")

DecisionTreeClassifier
real1: F1 Score = 0.756, Precision = 0.676, Recall = 0.859
real2: F1 Score = 0.800, Precision = 0.800, Recall = 0.800
syn1: F1 Score = 0.918, Precision = 0.867, Recall = 0.974
syn2: F1 Score = 0.888, Precision = 0.822, Recall = 0.966
syn3: F1 Score = 0.951, Precision = 0.961, Recall = 0.941
syn4: F1 Score = 0.903, Precision = 0.934, Recall = 0.873
syn5: F1 Score = 0.989, Precision = 1.000, Recall = 0.978


In [188]:
# Training and testing the random forest classifier
rf_clf = RandomForestClassifier(random_state=88)
rf_clf.fit(preprocess_features(X_train_all), proprocess_target(y_train_all))

print("RandomForestClassifier")
for name in split_data:
    y_pred = rf_clf.predict(preprocess_features(split_data[name]["X_test"]))
    f1 = f1_score(split_data[name]["y_test"], y_pred)
    precision = precision_score(split_data[name]["y_test"], y_pred)
    recall = recall_score(split_data[name]["y_test"], y_pred)

    # Print formatted results
    print(f"{name}: F1 Score = {f1:.3f}, Precision = {precision:.3f}, Recall = {recall:.3f}")

RandomForestClassifier
real1: F1 Score = 0.855, Precision = 0.813, Recall = 0.902
real2: F1 Score = 0.876, Precision = 0.937, Recall = 0.822
syn1: F1 Score = 0.932, Precision = 0.891, Recall = 0.977
syn2: F1 Score = 0.910, Precision = 0.851, Recall = 0.977
syn3: F1 Score = 0.962, Precision = 0.974, Recall = 0.951
syn4: F1 Score = 0.918, Precision = 0.961, Recall = 0.878
syn5: F1 Score = 0.991, Precision = 0.999, Recall = 0.982


## Naive Approach: Intersection of Top 2 Callers

In [ ]:
# Check performance of individual callers
def mutect2_predictor(X):
    return X["FILTER_Mutect2"] == True

y_pred = mutect2_predictor(X_test_all)
f1 = f1_score(y_test_all, y_pred)
precision = precision_score(y_test_all, y_pred)
recall = recall_score(y_test_all, y_pred)
print(f"Mutect2: F1 Score = {f1:.3f}, Precision = {precision:.3f}, Recall = {recall:.3f}")

def freebayes_predictor(X):
    return X["FILTER_Freebayes"] == True

y_pred = freebayes_predictor(X_test_all)
f1 = f1_score(y_test_all, y_pred)
precision = precision_score(y_test_all, y_pred)
recall = recall_score(y_test_all, y_pred)
print(f"FreeBayes: F1 Score = {f1:.3f}, Precision = {precision:.3f}, Recall = {recall:.3f}")

def vardict_predictor(X):
    return X["FILTER_Vardict"] == True

y_pred = vardict_predictor(X_test_all)
f1 = f1_score(y_test_all, y_pred)
precision = precision_score(y_test_all, y_pred)
recall = recall_score(y_test_all, y_pred)
print(f"Vardict: F1 Score = {f1:.3f}, Precision = {precision:.3f}, Recall = {recall:.3f}")

def varscan_predictor(X):
    return X["FILTER_Varscan"] == True

y_pred = varscan_predictor(X_test_all)
f1 = f1_score(y_test_all, y_pred)
precision = precision_score(y_test_all, y_pred)
recall = recall_score(y_test_all, y_pred)
print(f"Varscan: F1 Score = {f1:.3f}, Precision = {precision:.3f}, Recall = {recall:.3f}")

Mutect2: F1 Score = 0.801, Precision = 0.690, Recall = 0.954
FreeBayes: F1 Score = 0.772, Precision = 0.684, Recall = 0.886
Vardict: F1 Score = 0.781, Precision = 0.662, Recall = 0.952
Varscan: F1 Score = 0.345, Precision = 0.241, Recall = 0.603


In [203]:
# Using the intersection of the top 2 callers: Mutect2 and Vardict

def mutect2_vardict_predictor(X):
    return (X["FILTER_Mutect2"] == True) & (X["FILTER_Vardict"] == True)

y_pred = mutect2_vardict_predictor(X_test_all)
f1 = f1_score(y_test_all, y_pred)
precision = precision_score(y_test_all, y_pred)
recall = recall_score(y_test_all, y_pred)
print(f"Mutect2 + Vardict: F1 Score = {f1:.3f}, Precision = {precision:.3f}, Recall = {recall:.3f}")

for name in split_data:
    y_pred = mutect2_vardict_predictor(preprocess_features(split_data[name]["X_test"]))
    f1 = f1_score(split_data[name]["y_test"], y_pred)
    precision = precision_score(split_data[name]["y_test"], y_pred)
    recall = recall_score(split_data[name]["y_test"], y_pred)

    # Print formatted results
    print(f"{name}: F1 Score = {f1:.3f}, Precision = {precision:.3f}, Recall = {recall:.3f}")

Mutect2 + Vardict: F1 Score = 0.946, Precision = 0.967, Recall = 0.926
real1: F1 Score = 0.838, Precision = 0.823, Recall = 0.855
real2: F1 Score = 0.781, Precision = 0.735, Recall = 0.833
syn1: F1 Score = 0.931, Precision = 0.890, Recall = 0.976
syn2: F1 Score = 0.908, Precision = 0.847, Recall = 0.979
syn3: F1 Score = 0.962, Precision = 0.963, Recall = 0.961
syn4: F1 Score = 0.888, Precision = 0.957, Recall = 0.828
syn5: F1 Score = 0.972, Precision = 0.999, Recall = 0.946
